# Meet the data: outpatient appointments at Riverbend Community Hospital

**Master in Healthcare Analytics · Dataset 1 of 5 · Introduction**

This notebook introduces the first dataset we will work with throughout the programme. It has four parts: the clinical and operational context the data comes from, how to load it, a first look at the rows, and a column-by-column explanation of what each field means and where it would come from in a real hospital.

Nothing here requires prior programming experience. Every code cell is short and its purpose is explained just above it.

## 1. Context: where this data comes from

### The hospital

Riverbend Community Hospital is a **fictional 300-bed community hospital in a mid-sized US city**. Like most US hospitals it runs an outpatient (ambulatory) service alongside its inpatient wards and emergency department. This dataset covers only the outpatient side: the appointments booked in four clinics that share one scheduling system.

| Clinic | Providers | What it sees |
|---|---|---|
| Family Medicine | 8 | Primary care for all ages (adults only in this dataset): chronic disease follow-up, acute problems, preventive visits |
| Internal Medicine | 7 | Adult primary care and general medical consultation, typically more complex or older patients |
| Cardiology | 5 | Specialty referrals: hypertension, heart failure, arrhythmia, post-event follow-up |
| Orthopedics | 5 | Specialty referrals: joint, spine and musculoskeletal problems, pre- and post-surgical visits |

Clinics run **Monday to Friday, 08:00 to 16:30**, in 30-minute slots, with no appointments over lunch (12:00 and 12:30) and none on US federal holidays. The **providers** are physicians (MD, DO) and advanced-practice clinicians (NP, PA), some full-time and some part-time.

### The scheduling process

Every row in the main table is one **scheduled appointment**. The life of an appointment is simple:

1. A patient (or a referring clinician) **books** a slot with a provider on a future date. The number of days between the booking and the appointment is the **lead time**.
2. On the day, one of four things happens. The patient **arrives**; the patient **does not arrive and did not cancel** (a *no-show*); the patient **cancelled** in advance; or the **clinic cancelled** (provider sick, clinic closed, rescheduled).

### Why this matters clinically and operationally

No-shows are one of the most persistent problems in ambulatory care. In US clinics, somewhere between 10% and 30% of booked appointments end as no-shows. Each one is a missed opportunity for care (a delayed diagnosis, an uncontrolled chronic condition, a follow-up that never happens), an empty slot that another patient could have used, and lost revenue for the clinic. Understanding *which* appointments are likely to be missed, and *why*, is the first step to doing something about it: reminder calls, telehealth options, shorter waiting lists, or smarter overbooking.

This dataset was built around that question. Every variable in it is either a plausible driver of no-shows, a plausible red herring, or a key needed to link the tables together.

### What the data is, and is not

The data is **fully synthetic**. No row describes a real patient, provider or visit. It was generated from a statistical model so that the patterns in it are realistic in size and direction, but there is no privacy risk in using it, and there are no data-quality problems to fix: no missing values, no duplicates, no malformed dates. Patients are identified only by a pseudonymous ID; there are no names, birth dates, addresses or ZIP codes.

That said, everything in it corresponds to something a real hospital records. The note under each column below says where the field would come from in a real electronic health record (EHR) or practice-management system.

## 2. Loading the data

The dataset is three comma-separated files (CSV). We load them with **pandas**, the standard Python library for tables. `pd.read_csv` reads a file into a **DataFrame**, which you can think of as a spreadsheet held in memory: rows are records, columns are fields.

The `parse_dates` argument tells pandas that two columns hold dates, so it stores them as dates rather than text.

In [1]:
import pandas as pd

appointments = pd.read_csv("https://raw.githubusercontent.com/thousandoaks/Maths4DS-I/refs/heads/main/data/appointments.csv", parse_dates=["booking_date", "appointment_date"])
patients = pd.read_csv("https://raw.githubusercontent.com/thousandoaks/Maths4DS-I/refs/heads/main/data/patients.csv")
providers = pd.read_csv("https://raw.githubusercontent.com/thousandoaks/Maths4DS-I/refs/heads/main/data/providers.csv")

print("appointments:", appointments.shape[0], "rows x", appointments.shape[1], "columns")
print("patients:    ", patients.shape[0], "rows x", patients.shape[1], "columns")
print("providers:   ", providers.shape[0], "rows x", providers.shape[1], "columns")

appointments: 79253 rows x 18 columns
patients:     11523 rows x 7 columns
providers:    25 rows x 5 columns


Three tables, three different **grains** (what one row represents):

| File | One row is... | Rows |
|---|---|---|
| `appointments.csv` | one scheduled appointment | 79,253 |
| `patients.csv` | one patient | 11,523 |
| `providers.csv` | one clinician | 25 |

`appointments.csv` is the table we will analyse most. It already contains the patient's age, sex, insurance and distance on every row, so it can be used on its own. The other two are reference tables, useful when we want to count patients rather than appointments, or compare providers.

## 3. A first look: the first 10 rows

`.head(10)` shows the first ten rows. The table is sorted by appointment date and time, so these are the first appointments of 2 January 2025 (1 January is a holiday). Scroll horizontally to see all 18 columns.

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
appointments.head(10)

,appointment_id,patient_id,provider_id,clinic,visit_type,booking_date,appointment_date,appointment_time,weekday,slot_period,lead_time_days,prior_no_shows,sex,age,primary_payer,residence_type,distance_miles,status
0,A000001,P000240,DR023,Orthopedics,Established,2024-12-30,2025-01-02,08:00,Thursday,Early morning,3,4,M,46,Commercial,Suburban,9.6,Cancelled by clinic
1,A000002,P000663,DR023,Orthopedics,Annual wellness,2024-12-11,2025-01-02,08:00,Thursday,Early morning,22,0,F,40,Commercial,Urban,1.2,Arrived
2,A000003,P001386,DR024,Orthopedics,Telehealth,2024-12-20,2025-01-02,08:00,Thursday,Early morning,13,0,F,42,Commercial,Rural,6.5,Arrived
3,A000004,P001895,DR021,Orthopedics,Established,2024-12-28,2025-01-02,08:00,Thursday,Early morning,5,1,M,56,Commercial,Urban,1.8,Arrived
4,A000005,P002435,DR007,Family Medicine,New patient,2024-12-25,2025-01-02,08:00,Thursday,Early morning,8,0,M,53,Commercial,Rural,17.4,Cancelled by patient
5,A000006,P002619,DR014,Internal Medicine,Established,2024-12-30,2025-01-02,08:00,Thursday,Early morning,3,2,M,18,Commercial,Suburban,10.4,Arrived
6,A000007,P003565,DR004,Family Medicine,New patient,2024-12-24,2025-01-02,08:00,Thursday,Early morning,9,2,F,73,Commercial,Urban,2.1,Arrived
7,A000008,P004215,DR004,Family Medicine,New patient,2024-12-08,2025-01-02,08:00,Thursday,Early morning,25,0,M,82,Commercial,Urban,2.5,Arrived
8,A000009,P004290,DR008,Family Medicine,Telehealth,2024-12-29,2025-01-02,08:00,Thursday,Early morning,4,0,M,40,Commercial,Suburban,7.9,Arrived
9,A000010,P004499,DR006,Family Medicine,Telehealth,2024-12-22,2025-01-02,08:00,Thursday,Early morning,11,2,F,37,Commercial,Rural,56.1,Arrived


A few things to notice before we go column by column:

- Every row has an `appointment_id` (unique) and a `patient_id` (which repeats across rows, because patients come back).
- The dates are real calendar dates; `lead_time_days` is simply the gap between them.
- The last column, `status`, is the outcome. Most rows say *Arrived*; some say *No-show*.
- The patient-level columns on the right (`sex`, `age`, `primary_payer`, ...) describe the person, not the visit.

It is also worth looking at the two reference tables once:

In [3]:
patients.head(10)

,patient_id,sex,age,primary_payer,residence_type,distance_miles,home_clinic
0,P000001,M,18,Commercial,Suburban,9.2,Family Medicine
1,P000002,M,52,Commercial,Urban,5.2,Cardiology
2,P000003,M,42,Medicaid,Urban,4.0,Internal Medicine
3,P000004,F,32,Medicaid,Urban,2.3,Family Medicine
4,P000005,F,18,Medicaid,Urban,3.9,Internal Medicine
5,P000006,F,66,Medicare,Suburban,12.4,Family Medicine
6,P000007,M,38,Medicaid,Suburban,12.5,Family Medicine
7,P000008,F,57,Commercial,Urban,2.3,Family Medicine
8,P000009,M,65,Medicare,Rural,9.7,Orthopedics
9,P000010,M,57,Commercial,Suburban,11.9,Cardiology


In [4]:
providers.head(10)

,provider_id,clinic,role,fte,years_in_practice
0,DR001,Family Medicine,MD,1.0,24
1,DR002,Family Medicine,MD,0.8,28
2,DR003,Family Medicine,DO,1.0,4
3,DR004,Family Medicine,MD,1.0,6
4,DR005,Family Medicine,MD,1.0,8
5,DR006,Family Medicine,PA,1.0,8
6,DR007,Family Medicine,MD,1.0,27
7,DR008,Family Medicine,NP,0.8,8
8,DR009,Internal Medicine,MD,0.6,25
9,DR010,Internal Medicine,MD,0.8,24


## 4. The columns, one by one

For each column: what it means, what values it takes, where it comes from in a real hospital, and why it might matter for the no-show question. The small code cell after each group shows the actual values in the data.

### 4.1 Identifiers: who, with whom, where

**`appointment_id`** — A unique code for each scheduled appointment (`A000001`, `A000002`, ...). Assigned in date order. This is the table's *primary key*: no two rows share it. In a real system this is the encounter or visit ID generated by the scheduling module.

**`patient_id`** — A pseudonymous code for the patient (`P000001`, ...). The same patient appears on every appointment they had, so this value repeats. It links to `patients.csv`. In a real hospital this corresponds to the **medical record number (MRN)**, which would never leave the institution un-pseudonymised.

**`provider_id`** — The clinician the patient was booked with (`DR001` to `DR025`). Links to `providers.csv`, which gives the clinician's clinic, role and hours. Real systems use a provider ID or the national NPI number.

**`clinic`** — Which of the four clinics the appointment was in: Family Medicine, Internal Medicine, Cardiology or Orthopedics. A provider works in exactly one clinic. Specialty clinics tend to have longer waiting lists than primary care, which matters for lead time.

In [5]:
print("Unique appointments:", appointments["appointment_id"].nunique())
print("Unique patients:    ", appointments["patient_id"].nunique())
print("Unique providers:   ", appointments["provider_id"].nunique())
print()
print("Appointments per clinic:")
print(appointments["clinic"].value_counts())

Unique appointments: 79253
Unique patients:     11523
Unique providers:    25

Appointments per clinic:
clinic
Family Medicine      30563
Internal Medicine    24540
Cardiology           12528
Orthopedics          11622
Name: count, dtype: int64


### 4.2 The visit

**`visit_type`** — The kind of visit that was booked:

- *Established* — a follow-up or routine visit with a patient the clinic already knows (about 55% of appointments).
- *New patient* — the patient's first visit to that clinic, often on referral (about 20%). New-patient visits are longer, are booked further ahead, and in many clinics are missed more often.
- *Annual wellness* — a preventive visit (the Medicare Annual Wellness Visit or a commercial-plan physical), about 12%.
- *Telehealth* — a video visit, about 13%, more common in the primary-care clinics. No travel is needed, which is relevant to no-shows.

In the EHR this is the appointment type or visit reason chosen at booking.

In [6]:
appointments["visit_type"].value_counts()

visit_type
Established        43233
New patient        15820
Telehealth         10724
Annual wellness     9476
Name: count, dtype: int64

### 4.3 When: dates and times

**`booking_date`** — The date the appointment was made. Bookings for early January were made in late 2024, so this column starts in September 2024.

**`appointment_date`** — The scheduled date of the visit. All in 2025, weekdays only, no federal holidays.

**`appointment_time`** — The scheduled start of the slot, from 08:00 to 16:30 in 30-minute steps, skipping 12:00 and 12:30. Stored as text (`"08:00"`).

**`weekday`** — Monday to Friday, derived from `appointment_date`. Provided for convenience; you could compute it yourself.

**`slot_period`** — A three-level grouping of `appointment_time`, also derived: *Early morning* (08:00–09:30), *Mid-day* (10:00–14:30), *Late afternoon* (15:00–16:30). Grouping times into bands is a common analytical simplification: it is easier to compare three bands than sixteen slots.

**`lead_time_days`** — The number of days between booking and appointment (`appointment_date − booking_date`). Zero means same-day. This is an important operational measure: it reflects how long patients wait for care, and it is the variable clinics can most directly influence (by opening more slots, or holding some back for short-notice booking). It ranges from 0 to 120 days with a median of 11.

In [7]:
print("Appointment dates from", appointments["appointment_date"].min().date(), "to", appointments["appointment_date"].max().date())
print("Booking dates from    ", appointments["booking_date"].min().date(), "to", appointments["booking_date"].max().date())
print()
print("Appointments by weekday:")
print(appointments["weekday"].value_counts().reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]))
print()
print("Appointments by slot period:")
print(appointments["slot_period"].value_counts())
print()
print("Lead time (days):")
print(appointments["lead_time_days"].describe().round(1))

Appointment dates from 2025-01-02 to 2025-12-31
Booking dates from     2024-09-04 to 2025-12-29

Appointments by weekday:
weekday
Monday       14971
Tuesday      16355
Wednesday    16906
Thursday     15977
Friday       15044
Name: count, dtype: int64

Appointments by slot period:
slot_period
Mid-day           39378
Late afternoon    19963
Early morning     19912
Name: count, dtype: int64

Lead time (days):
count    79253.0
mean        15.7
std         15.8
min          0.0
25%          6.0
50%         11.0
75%         19.0
max        120.0
Name: lead_time_days, dtype: float64


### 4.4 The patient's history

**`prior_no_shows`** — How many appointments this patient missed (no-show) in the **365 days before** this appointment. It is a running count: it can change from one appointment to the next for the same patient, and it includes no-shows from late 2024 that are not themselves in the file. About half of appointments belong to patients with no recent no-shows; a small group of patients have five or more.

In a real system this would be computed from the patient's appointment history. It is usually the strongest single predictor of whether the next appointment will be missed, which is why reminder programmes often target patients with a history of missed visits.

In [8]:
appointments["prior_no_shows"].value_counts().sort_index().head(8)

prior_no_shows
0    38420
1    19453
2     9493
3     4317
4     2417
5     1390
6      822
7      563
Name: count, dtype: int64

### 4.5 The patient: demographics, coverage and access

These five columns describe the **patient**, not the visit, and are identical on every appointment the patient has. When you want to describe the patient population (for example, the age distribution), count each patient once by using `patients.csv` instead.

**`sex`** — F or M, as recorded at registration. About 54% of appointments are for women, which is typical of ambulatory care.

**`age`** — Age in whole years on 1 January 2025. Adults only (18 to 95). In a real dataset age would be derived from date of birth, which is itself a protected identifier under HIPAA and would not be released.

**`primary_payer`** — The patient's main insurance coverage. This is a distinctly US variable and one that clinicians should understand:

- *Commercial* — employer-sponsored or individually purchased private insurance (about half of appointments).
- *Medicare* — the federal programme for people aged 65 and over (and some younger people with disabilities). Almost all patients over 65 in the data are on Medicare.
- *Medicaid* — the state-federal programme for people on low incomes. Medicaid patients often face more barriers to attending: transport, work flexibility, and longer waits for appointments.
- *Self-pay* — no insurance; the patient pays directly.

Payer comes from the registration and billing system. It is *associated* with no-shows in most real datasets, but whether that is because of the coverage itself or because of the access barriers that go with it is exactly the kind of question this dataset lets you examine.

**`residence_type`** — Urban, Suburban or Rural, a broad description of where the patient lives. A stand-in for the ZIP code that a real dataset would hold (and that we omit for privacy).

**`distance_miles`** — Straight-line distance from the patient's home to the clinic, in miles, to one decimal place. Ranges from half a mile to 80 miles; the median is under 6 miles, but rural patients often live 20 miles or more away. Distance is a practical barrier to attendance and is routinely computed from the patient's address in real analyses.

In [9]:
print("Sex:")
print(appointments["sex"].value_counts())
print()
print("Primary payer:")
print(appointments["primary_payer"].value_counts())
print()
print("Residence type:")
print(appointments["residence_type"].value_counts())
print()
print("Age (years):")
print(appointments["age"].describe().round(1))
print()
print("Distance to clinic (miles):")
print(appointments["distance_miles"].describe().round(1))

Sex:
sex
F    42619
M    36634
Name: count, dtype: int64

Primary payer:
primary_payer
Commercial    41526
Medicare      17954
Medicaid      14047
Self-pay       5726
Name: count, dtype: int64

Residence type:
residence_type
Urban       36491
Suburban    29597
Rural       13165
Name: count, dtype: int64

Age (years):
count    79253.0
mean        52.6
std         17.4
min         18.0
25%         40.0
50%         53.0
75%         65.0
max         95.0
Name: age, dtype: float64

Distance to clinic (miles):
count    79253.0
mean         9.1
std          9.4
min          0.5
25%          3.2
50%          5.7
75%         11.4
max         80.0
Name: distance_miles, dtype: float64


### 4.6 The outcome

**`status`** — What happened to the appointment. Four values:

- *Arrived* — the patient attended.
- *No-show* — the patient neither attended nor cancelled. The slot was wasted.
- *Cancelled by patient* — the patient cancelled in advance. The slot could, in principle, be refilled.
- *Cancelled by clinic* — the clinic cancelled or rescheduled (provider unavailable, clinic closed).

This is the variable most of our later work will focus on. One convention matters from the start: the **no-show rate** is calculated as *No-show ÷ (Arrived + No-show)*, leaving cancellations out of the denominator, because a cancelled appointment never reached the day of the visit and so could not have been a no-show. Counting cancellations in the denominator understates the problem.

In [10]:
counts = appointments["status"].value_counts()
print(counts)
print()
print("Share of all appointments:")
print((counts / counts.sum() * 100).round(1).astype(str) + " %")
print()
kept = counts["Arrived"] + counts["No-show"]
print(f"No-show rate among appointments that were not cancelled: {counts['No-show'] / kept:.1%}")

status
Arrived                 61303
No-show                 10738
Cancelled by patient     5590
Cancelled by clinic      1622
Name: count, dtype: int64

Share of all appointments:
status
Arrived                 77.4 %
No-show                 13.5 %
Cancelled by patient     7.1 %
Cancelled by clinic      2.0 %
Name: count, dtype: object

No-show rate among appointments that were not cancelled: 14.9%


### 4.7 The reference tables in brief

**`patients.csv`** holds one row per patient with the same five patient columns as above, plus **`home_clinic`**: the clinic the patient attends most often. Roughly four in five of a patient's appointments are at their home clinic; the rest are referrals to another of the four clinics.

**`providers.csv`** holds one row per clinician: `provider_id`, `clinic`, **`role`** (MD = Doctor of Medicine, DO = Doctor of Osteopathic Medicine, NP = Nurse Practitioner, PA = Physician Assistant), **`fte`** (full-time equivalent: 1.0 full time, 0.8 or 0.6 part time, so part-time providers have proportionally fewer appointments) and **`years_in_practice`**.

In [11]:
providers.sort_values(["clinic", "provider_id"]).reset_index(drop=True)

,provider_id,clinic,role,fte,years_in_practice
0,DR016,Cardiology,NP,1.0,24
1,DR017,Cardiology,PA,1.0,13
2,DR018,Cardiology,MD,0.6,22
3,DR019,Cardiology,MD,1.0,9
4,DR020,Cardiology,MD,1.0,9
5,DR001,Family Medicine,MD,1.0,24
6,DR002,Family Medicine,MD,0.8,28
7,DR003,Family Medicine,DO,1.0,4
8,DR004,Family Medicine,MD,1.0,6
9,DR005,Family Medicine,MD,1.0,8


## 5. Summary

You now know what each of the 18 columns in `appointments.csv` means, where it would come from in a real hospital, and how the three files fit together. To recap the structure:

- **Keys**: `appointment_id` (unique), `patient_id` and `provider_id` (link to the reference tables).
- **The visit**: `clinic`, `visit_type`, `booking_date`, `appointment_date`, `appointment_time`, `weekday`, `slot_period`, `lead_time_days`.
- **The patient's history**: `prior_no_shows`.
- **The patient**: `sex`, `age`, `primary_payer`, `residence_type`, `distance_miles`.
- **The outcome**: `status`.

In the next sessions we will start describing these variables statistically, beginning with the continuous ones (age, lead time, distance), and work our way towards the question the dataset was built for: which appointments are likely to be missed, and what a clinic can do about it.